# Panda 100k Resume — Segment 2 (both arms, one notebook)

**Purpose.** Extend both the retrained baseline and the Koopman-ablated
model from their 50k checkpoints to 100k total steps, symmetrically, per
the pre-registered protocol (Log, Section 11 / Experiment 28 next steps).

**Pre-registered protocol (fixed before launch):**
1. **Symmetric resume** — both arms get the identical segment-2 recipe.
   Never run one arm without the other.
2. **Schedule discontinuity, documented:** the 50k checkpoints sit at the
   end of a fully decayed cosine schedule (warmup 10k, decay to ~0 at
   50k), weights-only (no optimizer state). Segment 2 therefore uses a
   fresh AdamW with a 2k-step re-warmup to peak LR 5e-5, cosine decay to
   ~0 over 50k steps. Rationale for 5e-5 (not a full 1e-4 warm restart):
   loss was still decreasing into the decayed LR at 50k, so headroom
   exists, but this is a *comparison* study — the lower-variance restart
   is preferred over maximum absolute progress. Identical for both arms,
   so internal validity of ablation-vs-baseline is preserved regardless.
3. **Loss curves saved to CSV every 500 steps** (discriminates the
   training-instability explanation for the 50k OOD pattern).
4. **Checkpoint every 10k steps** (session-crash insurance) + final.
5. **Arm-identity assertion:** the notebook refuses to train if the
   attached checkpoint's `training_info.json` disagrees with the ARM
   flag — prevents resuming the wrong arm's weights.
6. **No OOD conclusions from this notebook.** The quick in-distribution
   cell at the end is a smoke test only. The formal gate (Lorenz + 2
   held-out systems, n=20, direct paired Wilcoxon ablation-vs-baseline,
   raw predictions saved) runs in the separate evaluation notebook,
   in-distribution FIRST, before any OOD table is generated.

**Outcome mapping (from the Log):** Burgers-pattern persists at 100k
with the in-distribution gate passed -> lifting-matters rises to medium
confidence. OOD differences wash out -> 50k pattern attributed to
undertraining. In-distribution gap closes -> lifting hypothesis heavily
damaged.

**Launch prerequisite:** upload both 50k `checkpoint-final/` directories
as ONE private Kaggle Dataset with this layout, and attach it:
```
panda-50k-checkpoints/
  baseline/checkpoint-final/...
  koopman_ablation/checkpoint-final/...
```
Run this notebook twice: once with `ARM='baseline'`, once with
`ARM='ablation'` (~10h each at ~1.4 it/s; both fit one quota week).

## 0. Arm switch — the only line to change between the two runs

Everything else (checkpoint path, run name, dynamics-embedding flag) is
derived from `ARM`, so a flag/name/checkpoint mismatch is impossible by
construction, and the assertion in Section 3 double-checks against the
checkpoint's own metadata.

In [ ]:
# ============================================================
# ARM SWITCH - set 'baseline' for run 1, 'ablation' for run 2
# ============================================================
ARM = 'baseline'

assert ARM in ('baseline', 'ablation')
USE_DYNAMICS_EMBEDDING = (ARM == 'baseline')
RUN_NAME = 'baseline' if ARM == 'baseline' else 'koopman_ablation'

# Attached Kaggle Dataset holding the 50k weights (see header for layout)
RESUME_CKPT_DIR = f'/kaggle/input/panda-50k-checkpoints/{RUN_NAME}/checkpoint-final'

# Segment-2 schedule - PRE-REGISTERED, identical for both arms
SEGMENT_STEPS      = 50000
GLOBAL_STEP_OFFSET = 50000     # steps completed in segment 1
LR_SEGMENT2        = 5e-5
WARMUP2_STEPS      = 2000

print(f'ARM={ARM}  RUN_NAME={RUN_NAME}  use_dynamics_embedding={USE_DYNAMICS_EMBEDDING}')
print(f'Resuming from: {RESUME_CKPT_DIR}')

In [ ]:
import subprocess
subprocess.run(['pip', 'uninstall', 'peft', '-y'], capture_output=True)

# Then restart kernel again (skip Cell 1 again after restart)
import IPython
IPython.Application.instance().kernel.do_shutdown(restart=True)

## 1. Install dependencies

In [ ]:
# Install panda repo (architecture + training code)
!git clone --depth=1 https://github.com/abao1999/panda.git

# Install panda dependencies
# Note: panda uses uv but we install manually for Kaggle compatibility
%cd panda
!pip install -e . --quiet

# Additional dependencies needed for training
!pip install gluonts wandb --quiet

# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
print(f'GPU count: {torch.cuda.device_count()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')
        print(f'  VRAM: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB')
        print(f'  Compute capability: {torch.cuda.get_device_capability(i)}')

## 2. Load training data from HuggingFace

In [ ]:
from datasets import load_dataset
import numpy as np

print('Downloading GilpinLab/skew40 (~3GB)...')
hf_dataset = load_dataset('GilpinLab/skew40', split='train')
print(f'Loaded {len(hf_dataset)} trajectories')
print(f'Columns: {hf_dataset.column_names}')

# Inspect one example to verify format
example = hf_dataset[0]
target = np.array(example['target'])
print(f'\nExample trajectory shape: {target.shape}')  # expect [C, T]
print(f'Start: {example["start"]}')
print(f'Source dir: {example["_source_directory"]}')

In [ ]:
from gluonts.dataset.common import Dataset as GluonTSDataset
import pandas as pd
import numpy as np

print('Converting to pandas...')
df = hf_dataset.to_pandas()
print(f'Done. Shape: {df.shape}')

print('Extracting trajectories...')
targets = []
for i, row in df.iterrows():
    target = np.array(row['target'], dtype=np.float32)
    shape = row['target._np_shape']
    if shape is not None:
        target = target.reshape(shape)
    targets.append(target)

starts = df['start'].tolist()
print(f'Loaded {len(targets)} trajectories')
print(f'Sample shape: {targets[0].shape}')
print(f'Approx RAM: {sum(t.nbytes for t in targets) / 1e9:.2f} GB')

class InMemoryGluonDataset(GluonTSDataset):
    def __init__(self, targets, starts, freq='h'):
        self.targets = targets
        self.starts = starts
        self.freq = freq

    def __iter__(self):
        for target, start in zip(self.targets, self.starts):
            yield {
                'start': pd.Period(start, freq=self.freq),
                'target': target
            }

    def __len__(self):
        return len(self.targets)

test_item = next(iter(InMemoryGluonDataset(targets, starts)))
print(f'Wrapper test — shape: {test_item["target"].shape}, start: {test_item["start"]}')
print('Pre-load OK')

## 3. Model config — 21M baseline

In [ ]:
import sys
sys.path.insert(0, '/kaggle/working/panda')

from transformers import PatchTSTConfig
from panda.patchtst.patchtst import PatchTSTForPrediction
from panda.utils.train_utils import load_patchtst_model

# 21M model config — matches GilpinLab/panda checkpoint
# use_dynamics_embedding controlled by ablation flag above
MODEL_CONFIG = dict(
    mode='predict',
    context_length=512,
    prediction_length=128,
    patch_length=16,
    patch_stride=16,
    num_hidden_layers=8,
    d_model=512,
    num_attention_heads=8,
    channel_attention=True,
    ffn_dim=512,
    norm_type='rmsnorm',
    norm_eps=1e-5,
    attention_dropout=0.0,
    positional_dropout=0.0,
    path_dropout=0.0,
    ff_dropout=0.0,
    bias=True,
    activation_function='gelu',
    pre_norm=True,
    use_cls_token=False,
    init_std=0.02,
    scaling='std',
    pooling_type='max',
    head_dropout=0.0,
    # rope
    channel_rope=False,
    max_wavelength=500,
    rope_percent=0.75,
    # loss
    loss='mse',
    distribution_output=None,
    # Koopman lifting — controlled by ablation flag
    use_dynamics_embedding=USE_DYNAMICS_EMBEDDING,
    num_poly_feats=120,
    poly_degrees=2,
    rff_trainable=False,
    rff_scale=1.0,
    num_rff=256,
    # masking (unused in predict mode but required by config)
    do_mask_input=None,
    mask_type='random',
    random_mask_ratio=0.5,
    channel_consistent_masking=False,
    mask_value=0,
    num_forecast_mask_patches=3,
    unmasked_channel_indices=None,
    num_parallel_samples=100,
)

model = load_patchtst_model(
    mode='predict',
    model_config=MODEL_CONFIG,
    pretrained_encoder_path=None,
    pretained_checkpoint=None,
)

trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: use_dynamics_embedding={USE_DYNAMICS_EMBEDDING}')
print(f'Trainable parameters: {trainable_params:,}')
# Expect ~21M for baseline, slightly less for ablation (no RFF/poly weights)

# ============================================================
# RESUME: load 50k weights into the freshly built architecture
# ============================================================
import os, json

info_path = os.path.join(RESUME_CKPT_DIR, 'training_info.json')
if os.path.exists(info_path):
    with open(info_path) as f:
        _info = json.load(f)
    assert _info['use_dynamics_embedding'] == USE_DYNAMICS_EMBEDDING, (
        f"ARM MISMATCH: checkpoint use_dynamics_embedding="
        f"{_info['use_dynamics_embedding']} but ARM='{ARM}'. Wrong "
        f"checkpoint attached - stopping before any training.")
    print(f"Checkpoint identity verified: run_name={_info.get('run_name')}, "
          f"use_dynamics_embedding={_info['use_dynamics_embedding']}")
else:
    print('WARNING: training_info.json missing - arm identity NOT '
          'auto-verified. Confirm the dataset layout manually before '
          'proceeding.')

_st  = os.path.join(RESUME_CKPT_DIR, 'model.safetensors')
_bin = os.path.join(RESUME_CKPT_DIR, 'pytorch_model.bin')
if os.path.exists(_st):
    from safetensors.torch import load_file as _load_sf
    _state = _load_sf(_st)
elif os.path.exists(_bin):
    import torch as _t
    _state = _t.load(_bin, map_location='cpu')
else:
    raise FileNotFoundError(f'No weights found in {RESUME_CKPT_DIR}')

model.load_state_dict(_state, strict=True)
print(f'Loaded 50k weights (strict) from {RESUME_CKPT_DIR}')


## 4. Build training dataset

In [ ]:
import torch
from torch.utils.data import Dataset as TorchDataset
import numpy as np
import sys
sys.path.insert(0, '/kaggle/working/panda')

from panda.augmentations import (
    RandomTakensEmbedding,
    RandomConvexCombinationTransform,
    RandomAffineTransform,
    StandardizeTransform,
    RandomDimSelectionTransform,
)

SEED = 99

# Instantiate augmentations matching dataset.yaml
# probabilities [1.0, 1.0, 1.0, 0.0, 0.0] — first three active
aug_takens = RandomTakensEmbedding(lag_range=(1, 10), random_seed=SEED)
aug_convex = RandomConvexCombinationTransform(alpha=1.0, dim_range=(3, 8), random_seed=SEED)
aug_affine = RandomAffineTransform(scale=1.0, dim_range=(3, 8), random_seed=SEED)
standardize = StandardizeTransform()
dim_select = RandomDimSelectionTransform(num_dims=3, random_seed=SEED)

AUGMENTATION_RATE = 0.2  # from dataset.yaml

class PandaTrainDataset(TorchDataset):
    def __init__(self, targets, context_len=512, pred_len=128,
                 n_windows_per_traj=10, fixed_dim=3, seed=99):
        self.context_len = context_len
        self.pred_len = pred_len
        self.fixed_dim = fixed_dim
        self.rng = np.random.default_rng(seed)

        self.windows = []
        window_len = context_len + pred_len

        print('Pre-sampling windows with augmentations...')
        for i, target in enumerate(targets):
            C, T = target.shape
            if T < window_len:
                continue

            # Step 1: standardise full trajectory (StandardizeTransform)
            target = standardize(target, axis=-1)

            # Step 2: apply one augmentation with probability AUGMENTATION_RATE
            if self.rng.random() < AUGMENTATION_RATE:
                aug_idx = self.rng.integers(0, 3)  # choose one of three active augs
                try:
                    if aug_idx == 0:
                        target = aug_takens(target)
                    elif aug_idx == 1:
                        target = aug_convex(target)
                    else:
                        target = aug_affine(target)
                    # re-standardise after augmentation
                    target = standardize(target, axis=-1)
                except Exception:
                    pass  # if augmentation fails, use original

            # Step 3: select fixed_dim channels
            C_aug = target.shape[0]
            if C_aug > fixed_dim:
                ch_idx = self.rng.choice(C_aug, size=fixed_dim, replace=False)
                target = target[ch_idx]
            elif C_aug < fixed_dim:
                # pad by repeating channels if needed
                repeats = (fixed_dim // C_aug) + 1
                target = np.tile(target, (repeats, 1))[:fixed_dim]

            C_final, T_final = target.shape
            if T_final < window_len:
                continue

            # Step 4: sample windows
            max_start = T_final - window_len
            starts = np.linspace(0, max_start, n_windows_per_traj, dtype=int)
            for s in starts:
                self.windows.append(target[:, s:s+window_len].astype(np.float32))

            if (i + 1) % 5000 == 0:
                print(f'  Processed {i+1}/{len(targets)} trajectories...')

        self.windows = np.array(self.windows, dtype=np.float32)
        print(f'Pre-sampled {len(self.windows)} windows, shape: {self.windows.shape}')

    def __len__(self):
        return len(self.windows)

    def __getitem__(self, idx):
        window = self.windows[idx]  # [C, context+pred], already standardised
        ctx = window[:, :self.context_len]
        mean = ctx.mean(axis=-1, keepdims=True)
        std = ctx.std(axis=-1, keepdims=True).clip(min=1e-4)
        window_norm = ((window - mean) / std).clip(-5, 5)

        past = window_norm[:, :self.context_len].T
        future = window_norm[:, self.context_len:].T

        return {
            'past_values': torch.tensor(past, dtype=torch.float32),
            'future_values': torch.tensor(future, dtype=torch.float32),
        }

train_dataset = PandaTrainDataset(
    targets,
    context_len=512,
    pred_len=128,
    n_windows_per_traj=10,
    fixed_dim=3,
    seed=SEED
)

# Smoke test
sample = train_dataset[0]
print(f'past_values: {sample["past_values"].shape}')
print(f'future_values: {sample["future_values"].shape}')
print('Dataset OK')

## 5. Training

In [ ]:
import torch
import os, json, time
import pandas as pd
from torch.utils.data import DataLoader
from transformers import get_cosine_schedule_with_warmup

OUTPUT_DIR = f'/kaggle/working/{RUN_NAME}'
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 256
LOG_EVERY  = 500
SAVE_EVERY = 10000
GRAD_CLIP  = 1.0
SEED       = 99          # same as segment 1; identical across arms

torch.manual_seed(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    drop_last=True,
)

# Fresh AdamW (segment-1 optimizer state was not saved) + pre-registered
# segment-2 schedule: 2k warmup to 5e-5, cosine decay over 50k.
optimizer = torch.optim.AdamW(model.parameters(), lr=LR_SEGMENT2, weight_decay=0.0)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=WARMUP2_STEPS,
    num_training_steps=SEGMENT_STEPS,
)

scaler = torch.cuda.amp.GradScaler(init_scale=256)
model.train()

step = 0
data_iter = iter(loader)
losses = []
loss_rows = []
loss_csv = os.path.join(OUTPUT_DIR, f'loss_history_{RUN_NAME}_seg2.csv')

print(f'Run: {RUN_NAME} (segment 2)')
print(f'use_dynamics_embedding: {USE_DYNAMICS_EMBEDDING}')
print(f'Batch: {BATCH_SIZE}, Segment steps: {SEGMENT_STEPS}, '
      f'Global: {GLOBAL_STEP_OFFSET} -> {GLOBAL_STEP_OFFSET + SEGMENT_STEPS}')
print('Starting...\n')

t_start = time.time()
t_log = time.time()

while step < SEGMENT_STEPS:
    try:
        batch = next(data_iter)
    except StopIteration:
        data_iter = iter(loader)
        batch = next(data_iter)

    past = batch['past_values'].to(device)
    future = batch['future_values'].to(device)

    optimizer.zero_grad()
    with torch.autocast('cuda', dtype=torch.float16):
        out = model(past_values=past, future_values=future)
        loss = out.loss

    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    losses.append(loss.item())
    step += 1
    gstep = GLOBAL_STEP_OFFSET + step

    if step % LOG_EVERY == 0:
        elapsed = time.time() - t_log
        avg_loss = sum(losses[-LOG_EVERY:]) / LOG_EVERY
        its = LOG_EVERY / elapsed
        remaining = (SEGMENT_STEPS - step) / its / 3600
        print(f'step {gstep:>6}/{GLOBAL_STEP_OFFSET + SEGMENT_STEPS} | '
              f'loss {avg_loss:.4f} | lr {scheduler.get_last_lr()[0]:.2e} | '
              f'{its:.2f} it/s | ~{remaining:.1f}h remaining')
        loss_rows.append({'global_step': gstep, 'loss': avg_loss,
                          'lr': scheduler.get_last_lr()[0],
                          'elapsed_h': (time.time() - t_start) / 3600})
        pd.DataFrame(loss_rows).to_csv(loss_csv, index=False)
        t_log = time.time()

    if step % SAVE_EVERY == 0:
        ckpt_dir = os.path.join(OUTPUT_DIR, f'checkpoint-{gstep}')
        model.save_pretrained(ckpt_dir)
        print(f'  Saved checkpoint: {ckpt_dir}')

total_time = (time.time() - t_start) / 3600
print(f'\nSegment 2 done. Total time: {total_time:.2f}h')
print(f'Loss history: {loss_csv}')

## 6. Save final checkpoint

In [ ]:
import json, os

final_ckpt_dir = os.path.join(OUTPUT_DIR, 'checkpoint-100k-final')
model.save_pretrained(final_ckpt_dir)

training_info = {
    'run_name': RUN_NAME,
    'use_dynamics_embedding': USE_DYNAMICS_EMBEDDING,
    'total_steps': GLOBAL_STEP_OFFSET + SEGMENT_STEPS,
    'resumed_from': RESUME_CKPT_DIR,
    'segments': [
        {'steps': 50000, 'peak_lr': 1e-4,
         'schedule': 'cosine, warmup_ratio=0.2',
         'optimizer_state_carried': False},
        {'steps': SEGMENT_STEPS, 'peak_lr': LR_SEGMENT2,
         'schedule': f'cosine, warmup={WARMUP2_STEPS} steps',
         'optimizer': 'fresh AdamW, weight_decay=0.0'},
    ],
    'seed': 99,
    'model_config': MODEL_CONFIG,
}
with open(os.path.join(final_ckpt_dir, 'training_info.json'), 'w') as f:
    json.dump(training_info, f, indent=2)

print(f'Final 100k checkpoint: {final_ckpt_dir}')
print('Download checkpoint-100k-final/ AND the loss_history CSV.')
print('Formal evaluation (in-distribution gate first) happens in the')
print('separate eval notebook - not here.')

## 7. Quick in-distribution smoke test (NOT the formal gate)

This cell is a fast sanity read only. The formal in-distribution gate
(n=20 windows, both arms, direct paired Wilcoxon ablation-vs-baseline,
raw predictions saved, evaluated BEFORE any OOD table) runs in the
separate evaluation notebook once both arms' 100k checkpoints exist.

In [ ]:
from datasets import load_dataset as hf_load
from panda.patchtst.pipeline import PatchTSTPipeline
import numpy as np

# Load test split
hf_test = hf_load('GilpinLab/skew40', split='test')
test_example = hf_test[0]
target = np.array(test_example['target'])
if 'target._np_shape' in test_example and test_example['target._np_shape'] is not None:
    target = target.reshape(test_example['target._np_shape'])

# target shape: [C, T] — take first 512 as context
C, T = target.shape
context = target[:, :512].T  # [512, C] — pipeline expects [T, C]
context_tensor = torch.tensor(context, dtype=torch.float32)

# Load from saved checkpoint
pipeline = PatchTSTPipeline.from_pretrained(
    mode='predict',
    pretrain_path=final_ckpt_dir,
    device_map='cpu',
)

pred = pipeline.predict(context_tensor, 128, limit_prediction_length=False)
pred = pred.squeeze().cpu().numpy()

# Ground truth for comparison
truth = target[:, 512:640].T  # [128, C]

# Compute MAE as quick sanity metric
# Per-window normalise (same as our evaluation protocol)
ctx_mean = context.mean(axis=0, keepdims=True)
ctx_std = context.std(axis=0, keepdims=True) + 1e-8
pred_norm = (pred - ctx_mean) / ctx_std
truth_norm = (truth - ctx_mean) / ctx_std
mae = np.mean(np.abs(pred_norm - truth_norm))

print(f'Sanity check MAE (normalised, one test trajectory): {mae:.4f}')
print('If MAE < 1.0, training likely converged. If >> 1.0, check training logs.')

In [1]:
import os
for item in sorted(os.listdir('/kaggle/working/koopman_ablation/')):
    print(item)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/koopman_ablation/'